In [1]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [2]:
dp = '/data2/hratch/human_me/'
preprocess.create_environment(input_path = dp + 'inputs/', build_path = dp + 'build/', 
                  outdir = dp + 'processed/', n_cores=20)

In [3]:
from preprocess import correct_inputs 

full model

In [4]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/recon2_2.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')
# # # optional - only if you want to express non-machinery proteins
# # correct_inputs.check_non_machinery(nonmachinery_file = '/data2/hratch/human_me/input_files/non_machinery.txt')

# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = False, compress_mrna = False)

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'me_model.pickle', 'wb') as handle:
#     pickle.dump(me_model, handle)

toy model

In [5]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/toy_model.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                                 model_id = 'toy_me_model')

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'toy_me_model.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)



# Check

In [6]:
from expression import build_me_model
toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                unmodeled_protein_frac = None,
                                                model_id = 'toy_me_model')
jabba = True
if jabba:
    for r in toy_me_model.reactions:
        if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
            r._lower_bound = -1000
            r._upper_bound = 1000
            
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


Generate ubiquitin reactions for proteasomal degradation
Generate ribosome


../scripts/expression/gene_information.py:112 UserWarning: HGNC:10368: The letter X is in the protein sequence. Replacing with a random amino acid
  1%|          | 7/591 [00:00<00:09, 63.70it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:10<00:00, 54.79it/s]


...

Check reaction mass balances


 11%|█▏        | 1446/12650 [00:00<00:00, 14450.23it/s]

Check correct coupling of metabolic machinery


100%|██████████| 12650/12650 [00:06<00:00, 2025.88it/s]

Time to build: 4.2855351646741235 minutes


In [7]:
# toy_me_model.add_boundary(metabolite = toy_me_model.metabolites.get_by_id('h_c'), 
#                           type = 'demand')
sln, stat, _ = toy_me_model.solve_lp(mu_val =  1e-9)

../scripts/core/model.py:259 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 281.157 seconds with status 0


In [11]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in toy_me_model.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln[toy_me_model.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
protein_biomass_to_biomass,protein_biomass_to_biomass,4.530376e-10
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,3.649381e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
DNA_biomass_to_biomass,DNA_biomass_to_biomass,1.400000e-11


In [9]:
[r for r in toy_me_model.reactions if 'TRANSLATION_ELONGATIONc' in r.id][1].reaction

'mu/(98855.3094656939*mu + 1977.10618931388) HGNC:10404_mrna_c + 0.129314718055995/(98855.3094656939*mu + 1977.10618931388) HGNC:10404_mrna_deg_proxy + 3.50728116866483e6*mu  7.01456233732966e8 TRANSLATION_ELONGATIONc_complex_c + 31.62758551999923 biomass_tRNA + 25 charged_generic_A_trna_c + 5 charged_generic_C_trna_c + 12 charged_generic_D_trna_c + 10 charged_generic_E_trna_c + 11 charged_generic_F_trna_c + 43 charged_generic_G_trna_c + 4 charged_generic_H_trna_c + 17 charged_generic_I_trna_c + 24 charged_generic_K_trna_c + 19 charged_generic_L_trna_c + 7 charged_generic_M_trna_c + 4 charged_generic_N_trna_c + 16 charged_generic_P_trna_c + 6 charged_generic_Q_trna_c + 24 charged_generic_R_trna_c + 14 charged_generic_S_trna_c + 21 charged_generic_T_trna_c + 21 charged_generic_V_trna_c + 3 charged_generic_W_trna_c + 7 charged_generic_Y_trna_c + 293 gtp_c + 294 h2o_c --> HGNC:10404_unfolded_protein_c + 31.3502743800000 biomass_protein + 293 gdp_c + 293 generic_trna_c + 586 h_c + 293 pi_c

In [29]:
# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'working_version_' + str(0) + '.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)

# S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
# fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
# S.to_hdf(fn, key = str(0), mode = 'w')

In [11]:
# tol = max([abs(i) for i in res['no_dummy']['infeasible_reactions'].values()])
# print('Tolerance: {}'.format(tol))
# fail = {k:v for k,v in res['dummy']['infeasible_reactions'].items() if abs(v) >= tol}

# fail_ids = set(pd.Series(list(fail.keys())).apply(lambda x: x.split('_')[0]))
# fail_metabs = [m for m in res['dummy']['model'].metabolites if m.id.split('_')[0] in fail_ids]